In [18]:
from dotenv import load_dotenv
load_dotenv()

True

In [19]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(
    file_path="news.csv",
    encoding="utf-8"
)

documents = loader.load()

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200
)

chunks=splitter.split_documents(documents)

In [21]:
from langchain_huggingface import HuggingFaceEmbeddings

emb_model=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7944.20it/s]


In [22]:
from langchain_community.vectorstores import Chroma

vector_store=Chroma(
    embedding_function=emb_model,
    persist_directory="vectorstore",
    collection_name="sample"
)


In [23]:
retriever=vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":4}
)

In [24]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""You are a helpful fake news classification assistant.

You have to analyze the input news and the retrieved documents, then predict whether the news is FAKE or REAL.

Input News:
{inputnews}

Retrieved Documents:
{docs}
""",
    input_variables=["inputnews", "docs"]
)

In [25]:
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnablePassthrough,RunnableLambda


model=ChatGroq(model="openai/gpt-oss-120b")
parser=StrOutputParser()

In [26]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [41]:
query="modi is the precident of india"

chain = (
    {
        "inputnews": RunnablePassthrough(),
        "docs": retriever | RunnableLambda(format_docs)
    }
    | prompt
    | model
    | parser
)

In [42]:
chain.invoke(query)

'FAKE'